In [ ]:
"""eod_sale_product — Silver build. Conform every source to final column names + types.
 
This is where the original combine_sale_bill renaming + np.select enum decodes
live now. Gold then only joins dims, coalesces identity, ranges purchase price,
unions canceled, and writes.
"""
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import ArrayType, StringType, StructField, StructType
 
from ssv_data.config import VN_TZ_OFFSET_HOURS
from ssv_data.io.readers import Readers
from ssv_data.io.writers import write_delta
from ssv_data.schema.cast import fill_missing_columns
from ssv_data.schema.registry import SALE_ITEM_SCHEMA, PROMOTION_SCHEMA
from ssv_data.transforms.common import pivot_status_times
from ssv_data.transforms.scd import has_scd2_shape, scd2_apply, scd2_initial

In [ ]:
ACTION_AUDIT_SCHEMA = ArrayType(StructType([
    StructField("is_captured", StringType()),
    StructField("user_type", StringType()),
]))
 
# Top-level bill fields that may be absent on some docs -> add as typed null.
RAW_BILL_DEFAULTS = {
    "staffId": "str", "staffName": "str",
    "receivableSupplierId": "int", "receivableSupplierName": "str",
    "supplierCode": "str", "supplierName": "str", "storeName": "str",
    "customerGender": "int", "customerAgeRange": "str", "customerNationality": "int",
    "syncedWithSapTime": "date", "customerNote": "str",
    "originalVoucherCode": "str", "voucherCode": "str",
    "saleNormalItems": "str", "transactionPromotions": "str",
}

In [ ]:
def _parse_array(df, col, schema):
    """Parse a nested column to array<struct> regardless of how Bronze stored it
    (JSON string from a Copy/JDBC source, or array<struct> from mongo-spark)."""
    dt = dict(df.dtypes).get(col, "string")
    src = F.col(col) if dt.startswith("string") else F.to_json(F.col(col))
    return F.from_json(src, schema)

In [ ]:
def _conform_sale(ctx, table, is_return):
    df = fill_missing_columns(ctx.spark.read.table(table), RAW_BILL_DEFAULTS)
    df = (df
          .withColumn("item", F.explode_outer(_parse_array(df, "saleNormalItems", SALE_ITEM_SCHEMA)))
          .withColumn("sale_bill_time", F.col("documentDate") + F.expr(f"INTERVAL {VN_TZ_OFFSET_HOURS} HOURS"))
          .withColumn("posting_time", F.col("postingDate") + F.expr(f"INTERVAL {VN_TZ_OFFSET_HOURS} HOURS")))
    it = "item."
    df = df.select(
        F.col("_id").alias("transaction_id"),
        "sale_bill_time", "posting_time",
        F.col("staffId").alias("staff_id"),
        F.col("staffName").alias("staff_name"),
        F.col("paymentMethod").alias("payment_method_id"),
        F.col("deliveryOrderNo").alias("order_no"),
        F.col("storeCode").alias("store_id"),
        F.col("storeName").alias("store_name"),
        F.col("storeType").alias("store_type"),
        F.col("supplierCode").alias("supplier_id"),
        F.col("supplierName").alias("supplier_name"),
        F.col("receivableSupplierId").alias("payment_supplier_id"),
        F.col("receivableSupplierName").alias("payment_supplier_name"),
        F.col("customerGender").alias("gender_raw"),
        F.col("customerAgeRange").alias("age_raw"),
        F.col("customerNationality").alias("nat_raw"),
        F.col("syncedWithSapTime").alias("synced_sap_time"),
        F.col("originalVoucherCode").alias("voucher_code_raw"),
        F.col("customerNote").alias("customer_note_raw"),
        F.coalesce(F.col("cash"), F.lit(0.0)).cast("double").alias("cash"),
        F.coalesce(F.col("voucher"), F.lit(0.0)).cast("double").alias("voucher"),
        F.coalesce(F.col("receivableAmount"), F.lit(0.0)).cast("double").alias("receivableAmount"),
        F.coalesce(F.col("cashAtBank"), F.lit(0.0)).cast("double").alias("cashAtBank"),
        F.col(it + "productCode").alias("product_id"),
        F.col(it + "productName").alias("product_name"),
        F.col(it + "uom").alias("product_uom"),
        F.col(it + "uomSize").alias("product_uom_size"),
        F.col(it + "quantity").alias("quantity"),
        F.col(it + "totalAmount").alias("total_amount"),
        F.col(it + "totalAmountWithoutTax").alias("total_amount_no_tax"),
        F.col(it + "totalAmountByPromotion").alias("amt_promo"),
        F.col(it + "totalAmountWithoutTaxByPromotion").alias("amt_promo_notax"),
        F.col(it + "winPromotion").alias("win_raw"),
        F.col(it + "productGroupId").alias("product_group_id"),
        F.col(it + "productGroup").alias("product_group_name"),
        F.col(it + "productSubCategoryId").alias("product_sub_category_id"),
        F.col(it + "productSubCategory").alias("product_sub_category_name"),
        F.col(it + "productCategoryId").alias("product_category_id"),
        F.col(it + "productCategory").alias("product_category_name"),
        F.col(it + "productUomId").alias("product_uom_id"),
        F.col(it + "retailSellingPrice").alias("product_price"),
        F.col(it + "retailBusinessType").alias("retail_business_type"),
        F.col(it + "outputVatCode").alias("output_vat_code"),
    )
    df = (df
          .withColumn("final_amount", F.when(F.col("win_raw"), F.col("amt_promo")).otherwise(F.col("total_amount")))
          .withColumn("final_amount_no_tax", F.when(F.col("win_raw"), F.col("amt_promo_notax")).otherwise(F.col("total_amount_no_tax")))
          .drop("win_raw", "amt_promo", "amt_promo_notax"))
    if is_return:
        for c in ["cash", "voucher", "receivableAmount", "cashAtBank"]:
            df = df.withColumn(c, -F.col(c))
        df = df.withColumn("refund_transaction_id", F.col("customer_note_raw"))
    else:
        df = df.withColumn("refund_transaction_id", F.lit("Store"))
    return df.drop("customer_note_raw")

In [ ]:
def _build_sale_line(ctx):
    sale = _conform_sale(ctx, ctx.bronze("sale_bill"), is_return=False)
    ret = _conform_sale(ctx, ctx.bronze("sale_return_bill"), is_return=True)
    df = sale.unionByName(ret, allowMissingColumns=True)
    pm = F.col("payment_method_id")
    g, a, n = F.col("gender_raw"), F.col("age_raw"), F.col("nat_raw")
    return (df
            .withColumn("report_date", F.to_date("sale_bill_time"))
            .withColumn("transaction_time", F.col("sale_bill_time"))
            .withColumn("bill_amount", F.col("cash") + F.col("receivableAmount") + F.col("cashAtBank"))
            .withColumn("voucher_amount", F.col("voucher"))
            .withColumn("voucher_code_arr", F.split(F.coalesce(F.col("voucher_code_raw"), F.lit("")), ",")).drop("voucher_code_raw")
            .withColumn("transaction_type",
                        F.when(F.col("transaction_id").startswith("R"), F.lit("Refund")).otherwise(F.lit("Sale Transaction")))
            .withColumn("delivery_status", F.lit("completed"))
            .withColumn("channel_name",
                        F.when(F.lower(F.coalesce(F.col("order_no"), F.lit(""))).startswith("savyu"), F.lit("Savyu")).otherwise(F.lit("Store")))
            .withColumn("payment_method_name",
                        F.when(pm == 0, "cash").when(pm == 1, "card").when(pm == 2, "voucher").when(pm == 4, "e-wallet").otherwise("unknown"))
            .withColumn("customer_gender", F.when(g == 1, "Male").when(g == 2, "Female").otherwise("")).drop("gender_raw")
            .withColumn("customer_age_range", F.when(a == "2", "Student").when(a == "3", "Middle").when(a == "5", "Foreigner").otherwise("")).drop("age_raw")
            .withColumn("customer_nationality", F.when(n == 0, "Vietnamese").when(n == 1, "Asian").when(n == 2, "US_UK").when(n == 3, "Other").otherwise("")).drop("nat_raw")
            .withColumn("row_num_transaction_id", F.row_number().over(Window.partitionBy("transaction_id").orderBy(F.monotonically_increasing_id())))
            .withColumn("number_of_line_item", F.count("*").over(Window.partitionBy("transaction_id"))))

In [ ]:
def _build_promotion(ctx):
    sb = fill_missing_columns(ctx.spark.read.table(ctx.bronze("sale_bill")), {"transactionPromotions": "str"})
    p = sb.select("_id", F.explode_outer(_parse_array(sb, "transactionPromotions", PROMOTION_SCHEMA)).alias("p"))
    promo = (p.where(F.col("p.promotionName").isNotNull()).groupBy("_id")
             .agg(F.collect_set("p.promotionCode").alias("promotion_id_arr"),
                  F.collect_set("p.promotionName").alias("promotion_name_arr")))
    voucher = (p.where((F.col("p.voucherCode").isNotNull()) &
                       ((F.col("p.voucherType") == 0) | (F.col("p.products").contains("95010032"))))
               .groupBy("_id")
               .agg(F.sum("p.totalDiscountAmount").alias("total_voucher_amount"),
                    F.collect_set("p.promotionName").alias("voucher_name_arr")))
    return (promo.join(voucher, "_id", "full")
            .withColumnRenamed("_id", "transaction_id")
            .withColumn("promotion_partner_store", F.lit(None).cast("string")))  # TODO: ARRAY_PATTERN map
 
 
def _users(ctx):
    return (ctx.spark.read.table(ctx.bronze("users"))
            .select(F.col("user_id"),
                    F.col("phone").alias("customer_phone"),
                    F.col("full_name").alias("user_name"),
                    F.col("customer_code")))
 
 
def _build_partner_dim(ctx):
    return (ctx.spark.read.table(ctx.bronze("partner_store"))
            .join(ctx.spark.read.table(ctx.bronze("partner_sources")), "partner_source", "left")
            .select("partner_store_id", F.col("name").alias("partner_store")))

def _build_delivery(ctx):
    s = ctx.spark
    deliv = (s.read.table(ctx.bronze("delivery_orders"))
             .withColumn("user_id", F.coalesce(F.col("user_id").cast("long"), F.lit(0)))
             .dropDuplicates(["order_id"])
             .withColumnRenamed("transaction_no", "transaction_id"))
    audit = pivot_status_times(s.read.table(ctx.bronze("delivery_order_audits")),
                               "order_id", "status", "created_at", [1, 5, 6, 7, 8, 12, 13])
    audit = (audit
             .withColumn("8", F.least(F.col("5"), F.col("8")))  # cancel can pre-empt complete
             .withColumnRenamed("1", "delivery_created_time")
             .withColumnRenamed("5", "delivery_canceled_time")
             .withColumnRenamed("6", "delivery_driver_booked_time")
             .withColumnRenamed("7", "delivery_picked_time")
             .withColumnRenamed("8", "delivery_completed_time")
             .withColumnRenamed("12", "delivery_store_delivered_time")
             .withColumnRenamed("13", "delivery_partial_completed_time"))
    # TODO: 7->12->7 first-pick + first-cancel nuance if SLA metrics depend on it.
    return (deliv.join(audit, "order_id", "left")
            .withColumn("channel_name",
                        F.when(F.col("external_source") != "", F.col("external_source"))
                         .when(F.col("order_id").isNotNull(), F.lit("7Now")).otherwise(F.lit("Unknown")))
            .withColumnRenamed("created_at", "dom_created_at")
            .withColumnRenamed("partner_order_no", "short_order_id")
            .join(_users(ctx), "user_id", "left")
            .withColumn("customer_id", F.col("user_id"))
            .join(_build_partner_dim(ctx), "partner_store_id", "left"))

def _build_transaction(ctx):
    st = ctx.spark.read.table(ctx.bronze("sale_transactions"))
    return (st.join(_users(ctx), "user_id", "left")
            .select(F.col("transaction_id"),
                    F.col("user_id").alias("customer_id"),
                    "customer_phone", "user_name", "customer_code",
                    "rating", "comment",
                    F.col("created_at").alias("dom_created_at")))
 
 
def _build_point_history(ctx):
    s = ctx.spark
    ph = s.read.table(ctx.bronze("point_histories"))
    ph = (ph.withColumn("aa", F.explode_outer(_parse_array(ph, "action_audits", ACTION_AUDIT_SCHEMA)))
            .withColumn("is_captured", F.col("aa.is_captured"))
            .withColumn("capture_method", F.col("aa.user_type")))
    w = Window.partitionBy("reference_id").orderBy(F.col("is_captured").desc_nulls_last())
    ph = ph.withColumn("rn", F.row_number().over(w)).where("rn = 1")
    return (ph.join(_users(ctx), "user_id", "left")
            .select(F.col("reference_id").alias("transaction_id"),
                    F.col("user_id").alias("customer_id"),
                    "customer_phone", "user_name", "customer_code", "capture_method",
                    F.col("created_at").alias("dom_created_at")))

def _build_dims(ctx):
    r = Readers(ctx)
    dim_product = (r.table("bronze", "product_uom_upc")
                   .where((F.col("status") == 1) & (F.col("is_base_unit") == 1))
                   .withColumnRenamed("product_uom_code", "product_uom")
                   .dropDuplicates(["product_id"]))
    dim_oms_product = (r.table("bronze", "oms_product")
                       .select("product_id", "product_brand_name", "product_manufacturer_name")
                       .dropDuplicates(["product_id"]))
    dim_store = (r.table("bronze", "oms_store")
                 .join(r.table("bronze", "dict_stores"), "store_id", "left")
                 .select("store_id", "store_name", "store_type", "tier_store", "sale_area_id")
                 .dropDuplicates(["store_id"]))
    dim_payment = r.table("bronze", "mapping_payment_method").dropDuplicates(["payment_method_id"])
    dim_partner_store = _build_partner_dim(ctx)
    dim_customer = _users(ctx).withColumnRenamed("user_id", "customer_id")
    dim_purchase_price = r.table("bronze", "purchase_price_timeline").dropDuplicates()
    return dict(dim_product=dim_product, dim_oms_product=dim_oms_product, dim_store=dim_store,
                dim_payment=dim_payment, dim_partner_store=dim_partner_store,
                dim_customer=dim_customer, dim_purchase_price=dim_purchase_price)

In [ ]:
# SCD2 policy: dims that accumulate history (valid_from / valid_to / is_current).
# Gold joins them as-of transaction time; BI consumers must filter is_current for
# the current view. All other dims stay SCD1 truncate-reload.
SCD2_DIMS = {"dim_product": ["product_id"], "dim_store": ["store_id"]}
 
 
def _write_scd2_dim(ctx, name, incoming, keys):
    """Accumulate SCD2 state. Existing SCD2 table -> scd2_apply (forward-only: an
    old-day backfill leaves the dim untouched). Missing table or legacy SCD1 shape
    -> scd2_initial bootstrap (auto-migration, valid_from at the epoch)."""
    tbl = ctx.silver(name)
    if ctx.spark.catalog.tableExists(tbl) and has_scd2_shape(ctx.spark.read.table(tbl)):
        state = scd2_apply(ctx.spark.read.table(tbl), incoming, keys, ctx.running_date)
    else:
        state = scd2_initial(incoming, keys)
    # localCheckpoint breaks lineage from the table we are about to overwrite;
    # overwrite_schema lets the SCD1 -> SCD2 migration add the 3 version columns.
    write_delta(ctx, state.localCheckpoint(), tbl, overwrite_schema=True)
 
 
def silver_build(ctx) -> None:
    rw = ctx.replace_where
    write_delta(ctx, _build_sale_line(ctx), ctx.silver("sale_line"), partition_col="report_date", replace_where=rw)
    write_delta(ctx, _build_promotion(ctx), ctx.silver("promotion"))
    write_delta(ctx, _build_delivery(ctx), ctx.silver("delivery"))
    write_delta(ctx, _build_transaction(ctx), ctx.silver("transaction"))
    write_delta(ctx, _build_point_history(ctx), ctx.silver("point_history"))
    for name, df in _build_dims(ctx).items():
        if name in SCD2_DIMS:
            _write_scd2_dim(ctx, name, df, SCD2_DIMS[name])
        else:
            write_delta(ctx, df, ctx.silver(name))
    ctx.logger.info("silver build complete")